# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


# Notebook 6 — Adaptive SLS with Wei-Chen's smooth prediction and nonsmooth Drake execution

**Current integration reference. The uploaded copy includes a completed local run; see `../docs/supplied_nb06_run.md`. It was not rerun for publication.**

Use **Python (adaptive_sls)**. Save in `~/projects/adaptive_contact_linear_pusher/notebooks/`.
Cell 1 locates both repositories and launches a separate Python process in your existing
`quasistatic-contact-SLS` environment for Drake. Neither repository is edited.

**Loop:** smooth JAX prediction → adaptive SQP/SLS/ADMM plan → convergence and first-command
checks → actual `QuasidynamicLinearPusher(dynamics_smoothing=0)` transition → parameter
update from the observed position → replan. Six planned commands; only the first is executed
at each of six receding-horizon steps.

**What changed:**
- Prediction is the closed-form solution of Wei-Chen's *fixed-κ log-barrier pusher problem*,
  ported to JAX for differentiation. It is not the earlier mass-independent `SIGMA` surrogate.
- Observation comes from the C++ Drake system in the other process; there is no synthetic fallback.
- Theorem 1 / equations (6)–(7) supply the smoothing-error enclosure. A conservative,
  zero-centered envelope covers **all masses in the prior and all contact excitations**.
- Numerical solve error has its own explicit budget, checked against the exact nonsmooth
  one-contact solution on every executed transition. Optional extra process error is separate.
- Fixed-κ dynamics are nonlinear in the parameter. The estimator includes a parameter Taylor
  allowance; the planner additionally uses an analytic state/input/parameter Taylor bound.
- Constraint rows follow the adaptive repository's required physical-state-box/input-box layout.

**Scope:** one-dimensional, mass-only, fixed-step quasidynamics; known stiffness and geometry;
exact observed state, object-position-only parameter update. No table friction, inertial velocity,
hardware, or sensor noise. The smoothing bound is analytic; the numerical budget is a checked
assumption, not a global solver-accuracy theorem. Information gain remains the repository's
lagged-generator surrogate, not a guarantee of a measured interval reduction.

**Sources used**
- Wei-Chen paper: *Certified Gradient-Based Contact-Rich Manipulation via Smoothing-Error
  Reachable Tubes*, arXiv:2602.09368v2, Theorem 1 and equations (6)–(7).
- Contact repository: `multibody/quasidynamic/quasidynamic_linear_pusher.cc`,
  `multibody/quasidynamic/quasidynamic_differentiable_contact_model.cc`,
  `solvers/solve_conic_program.cc`.
- Adaptive repository: `src/gpu_sls/{generic_mpc_wrapper,sqp,gpu_sls,gpu_admm}.py`.
  Current source inspected at https://github.com/trustworthyrobotics/adaptive_sls.

Cells defining functions do not solve anything. Cell 8 contains optional audits for **you** to run;
Cell 10 solves a demonstration horizon; Cell 11 starts a **fresh** closed-loop experiment.


## Cell 1 — Paths, imports, provenance, and a narrowly scoped compatibility fix
Edit `REPO`, `CONTACT_REPO`, or `DRAKE_LAUNCH` if your installation differs.
The previous JAX int64/int32 ADMM diagnostic-index problem is handled in memory only, for
the exact known source statement. Other source/API changes fail visibly.


In [ ]:
import os, sys, time, json, inspect, subprocess, hashlib, tempfile, selectors, ast, textwrap
from pathlib import Path
from dataclasses import dataclass
from functools import partial
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt

REPO = Path(os.environ.get('ADAPTIVE_SLS_REPO', Path.home()/'projects/adaptive_sls_complete')).expanduser().resolve()
CONTACT_REPO = Path(os.environ.get('CONTACT_SLS_REPO', Path.home()/'projects/quasistatic-contact-SLS')).expanduser().resolve()
CONDA_EXE = os.environ.get('CONDA_EXE', str(Path.home()/'miniforge3/bin/conda'))
DRAKE_LAUNCH = [CONDA_EXE, 'run', '--no-capture-output', '-n', 'quasistatic-contact-SLS', 'python', '-u']
# Alternative, if this interpreter already imports the custom pydrake binding correctly:
# DRAKE_LAUNCH = ['/absolute/path/to/the/working/drake/python', '-u']

adaptive_files = ['src/gpu_sls/'+p for p in ['generic_mpc_wrapper.py','sqp.py','gpu_sls.py','gpu_admm.py']]
contact_files = ['multibody/quasidynamic/quasidynamic_linear_pusher.cc',
                 'multibody/quasidynamic/quasidynamic_differentiable_contact_model.cc',
                 'solvers/solve_conic_program.cc']
for root, names in [(REPO,adaptive_files),(CONTACT_REPO,contact_files)]:
    for name in names:
        if not (root/name).is_file():
            raise FileNotFoundError(f'Check the repository path: {root/name}')
os.environ['JAX_PLATFORMS'] = os.environ.get('NB06_PLATFORM','cpu')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
sys.path.insert(0,str(REPO/'src'))
import jax
jax.config.update('jax_enable_x64',True)
import jax.numpy as jnp
import gpu_sls
import gpu_sls.gpu_admm as admm_module
import gpu_sls.gpu_sls as sls_module
import gpu_sls.sqp as sqp_module
from gpu_sls.gpu_sls import (SLSConfig, observable_subspace_factorizer,
    adaptive_posterior_factors, girard_reduction_opt)
from gpu_sls.sqp import SQPConfig
from gpu_sls.gpu_admm import ADMMConfig
from gpu_sls.generic_mpc_wrapper import GenericMPCControllerWrapper

if not Path(gpu_sls.__file__).resolve().is_relative_to(REPO/'src'):
    raise RuntimeError('Wrong gpu_sls is loaded. Restart the kernel and run this cell first.')
if 'enable_information_cost' not in SLSConfig.__dataclass_fields__:
    raise RuntimeError('This checkout lacks the required adaptive information-cost interface.')

COMPATIBILITY_PATCH = 'not needed'
# Read the file rather than inspect an in-memory patched function on a cell rerun.
admm_file_source=(REPO/'src/gpu_sls/gpu_admm.py').read_text()
admm_tree=ast.parse(admm_file_source)
admm_node=next(n for n in admm_tree.body if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name=='constrained_solve')
admm_source=ast.get_source_segment(admm_file_source,admm_node)
old_line = 'primal_worst_flat_index = jnp.argmax(jnp.abs(primal_residual_flat))'
if old_line in admm_source and old_line+'.astype(jnp.int32)' not in admm_source:
    patched_source = admm_source.replace(old_line,old_line+'.astype(jnp.int32)')
    namespace = dict(admm_module.__dict__)
    exec(compile(textwrap.dedent(patched_source),'<nb06 ADMM diagnostic dtype fix>','exec'),namespace)
    admm_module.constrained_solve = namespace['constrained_solve']
    sls_module.constrained_solve = admm_module.constrained_solve
    sqp_module.constrained_solve = admm_module.constrained_solve
    COMPATIBILITY_PATCH = 'known ADMM diagnostic argmax cast to int32, in memory only'

def repo_provenance(root,names):
    try:
        commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
    except (OSError,subprocess.CalledProcessError):
        commit='archive / no git metadata'
    return dict(path=str(root),commit=commit,
                hashes={p:hashlib.sha256((root/p).read_bytes()).hexdigest() for p in names})
PROVENANCE = dict(adaptive=repo_provenance(REPO,adaptive_files),
                  contact=repo_provenance(CONTACT_REPO,contact_files),
                  compatibility_patch=COMPATIBILITY_PATCH)
wrapper_source=(REPO/'src/gpu_sls/generic_mpc_wrapper.py').read_text()
sls_source=(REPO/'src/gpu_sls/gpu_sls.py').read_text()
if '_admm_primal_worst_flat_index' not in wrapper_source:
    raise RuntimeError('This notebook requires the diagnostic-enabled adaptive checkout used in the later Notebook 6 runs.')
if 'physical_widths = h_ct[:, :n_x]' not in sls_source or 'state_box_rows = nominal_nc - 2 * n_u' not in sls_source:
    raise RuntimeError('Adaptive tube-width extraction changed. Review its constraint layout before using this adapter.')
print('Adaptive repository:',REPO)
print('Contact repository:',CONTACT_REPO)
print('Drake launch:',DRAKE_LAUNCH)
print('JAX:',jax.__version__,'| devices:',jax.devices())
print('Compatibility:',COMPATIBILITY_PATCH)


## Cell 2 — Experiment and separate error budgets
Scaled state $z=[P,O,R]$ contains pusher position, object position, and previous command.
$P=(p-(o_0-w))/L$, $O=(o-o_0)/L$, $U=(u-(o_0-w))/L$.
The augmented state is $s=[z,hat\theta]$.

$$\alpha(m)=\frac{K_a h^2}{m+K_a h^2},\quad
\alpha=\alpha_c+\alpha_r\theta,\quad \theta\in[-1,1].$$

`KAPPA` is the **physical log-barrier coefficient**, not the old length smoothing `SIGMA`.
The inspected C++ code switches to its nonsmooth solver below $10^{-8}$, so use
`KAPPA >= 1e-8` when comparing its smooth output with the JAX port.

The numerical budgets below are acceptance tolerances, not derived global guarantees.
Every Drake execution is audited before its observation is used for learning. Extra process
error defaults to zero; sensor noise remains unsupported rather than silently ignored.


In [ ]:
DT, KA, WIDTH, O_ORIGIN, LENGTH = 0.02, 1.0, 0.10, 0.20, 0.01
MASS_PRIOR = np.array([0.005,0.030])
alpha_from_mass = lambda m: KA*DT**2/(np.asarray(m)+KA*DT**2)
ALPHA_LO,ALPHA_HI = alpha_from_mass(MASS_PRIOR[::-1])
ALPHA_C,ALPHA_R = (ALPHA_LO+ALPHA_HI)/2,(ALPHA_HI-ALPHA_LO)/2
KAPPA = 1e-8
TAU = KAPPA/(KA*LENGTH**2)
assert KAPPA >= 1e-8
NX,NT,NU,NA = 3,1,1,4
HORIZON,EXECUTION_STEPS,QMAX = 6,6,8
INFO_WEIGHT = 3.0

DRAKE_NUMERICAL_BUDGET_M = np.array([25e-6,0.5e-6,0.0])
EXTRA_PROCESS_BOUND_M = np.array([0.0,0.0,0.0])
FLOAT_ALLOWANCE_M = np.array([1e-10,1e-10,0.0])
SENSOR_BOUND_M = 0.0
assert SENSOR_BOUND_M == 0.0, 'This notebook assumes exact observations.'
assert EXTRA_PROCESS_BOUND_M[2] == 0.0, 'The previous command is known exactly.'

MOVE_MAX,BACKWARD_ALLOWANCE = 8e-3/LENGTH,0.2e-3/LENGTH
DU_MAX,U_MIN,U_MAX = 12e-3/LENGTH,-2e-3/LENGTH,35e-3/LENGTH
A_MIN = 1e-3/LENGTH
FORCE_MAX,FORCE_SCALE = 25e-3,20e-3
MOVE_SCALE,COMMAND_SCALE = 2e-3/LENGTH,10e-3/LENGTH
W_COST = jnp.array([1.0,0.25,0.05])
GAIN_REG = 1e-7
CHECK_TOL = 2e-5
DYNAMICS_DEFECT_TOL = 2e-5
RUN_INTERFACE_AUDIT = True
RUN_DEMO = True
RUN_CLOSED_LOOP = True
RUN_COMPARISON = False
RPC_TIMEOUT_SECONDS = 120
INITIAL_PHYS = jnp.array([-0.005,0.0,-0.005])
INITIAL_THETA = jnp.array([0.0])
INITIAL_G = jnp.zeros((1,QMAX)).at[0,0].set(1.0)
C_MEAS = jnp.diag(jnp.array([0.0,1.0,0.0]))
OBSTACLES = jnp.empty((0,3))

def mass_from_alpha(alpha):
    return KA*DT**2*(1/np.asarray(alpha)-1)
def parameter_interval(theta_hat,G):
    return float(np.asarray(theta_hat).reshape(-1)[0])+np.array([-1.0,1.0])*float(np.abs(np.asarray(G)).sum())
def reported_mass_interval(theta_hat,G):
    lo,hi=parameter_interval(theta_hat,G)
    lo,hi=max(lo,-1.0),min(hi,1.0)
    if lo>hi:
        raise RuntimeError('Parameter set and physical prior have empty intersection.')
    return mass_from_alpha(ALPHA_C+ALPHA_R*np.array([hi,lo]))
print('Mass prior [g]:',MASS_PRIOR*1e3)
print('Nominal mass [g]:',mass_from_alpha(ALPHA_C)*1e3)
print('Barrier kappa:',KAPPA,'| dimensionless tau:',TAU)
print('Numerical acceptance budgets [um]:',DRAKE_NUMERICAL_BUDGET_M*1e6)
print('Extra process bounds [um]:',EXTRA_PROCESS_BOUND_M*1e6)
print('Information weight:',INFO_WEIGHT,'| exact observations; object-only identification')


## Cell 3 — Exact one-contact specialization of Wei-Chen's smooth dynamics
His C++ pusher supplies $P=\operatorname{diag}(K_a,m/h^2)$ and $J=[-1,1]$.
`SolveConicProgram` with positive $\kappa$ minimizes
$$\frac{K_a}{2}(p^+-u)^2+\frac{m}{2h^2}(o^+-o)^2
 -\kappa\log(o^+-p^+-w).$$
Stationarity gives $p^+=u-\lambda_\kappa/K_a$, $o^+=o+h^2\lambda_\kappa/m$.
Writing $a_m=u+w-o$, $c=1/K_a+h^2/m$ and using
$\lambda_\kappa(c\lambda_\kappa-a_m)=\kappa$ gives
$$\lambda_\kappa=\frac{a_m+\sqrt{a_m^2+4c\kappa}}{2c}.$$
In scaled coordinates, with $a=U-O$ and $\tau=\kappa/(K_aL^2)$:
$$g_\kappa(a,\alpha)=\frac{a+\sqrt{a^2+4\tau/(1-\alpha)}}{2},\qquad
\boxed{f_\kappa(z,U;\theta)=[U-(1-\alpha)g_\kappa,\ O+\alpha g_\kappa,\ U].}$$
The square-root term now depends on mass. `physical_step` is this exact mathematical
solution in JAX, not a call through C++ and not an approximation with fixed `SIGMA`.
`exact_nonsmooth_step` is used only for numerical audits and independent checks;
**it is never substituted for the Drake observation**.


In [ ]:
def response_alpha(theta):
    return ALPHA_C+ALPHA_R*jnp.ravel(theta)[0]

def barrier_excitation(a,alpha):
    return 0.5*(a+jnp.sqrt(a*a+4*TAU/(1-alpha)))

def physical_step(z,u,theta):
    alpha=response_alpha(theta)
    g=barrier_excitation(u[0]-z[1],alpha)
    return jnp.stack([u[0]-(1-alpha)*g,z[1]+alpha*g,u[0]])

def dynamics(s,u,t,*,parameter):
    del t,parameter
    return jnp.concatenate([physical_step(s[:NX],u,s[NX:]),s[NX:]])

def nominal_force(z,u,theta):
    alpha=response_alpha(theta)
    return KA*LENGTH*(1-alpha)*barrier_excitation(u[0]-z[1],alpha)

def exact_nonsmooth_step(z,u,theta):
    alpha=response_alpha(theta)
    a=jnp.maximum(u[0]-z[1],0.0)
    return jnp.stack([u[0]-(1-alpha)*a,z[1]+alpha*a,u[0]])

def weichen_E(z,u,theta):
    # Scaled form of -P^{-1} J^T (d lambda_kappa / d kappa) kappa.
    alpha=response_alpha(theta)
    a=u[0]-z[1]
    D=jnp.sqrt(a*a+4*TAU/(1-alpha))
    return TAU/D*jnp.array([1.0,-alpha/(1-alpha),0.0])

print('Fixed-kappa log-barrier prediction and theorem generator defined.')


## Cell 4 — Smoothing envelope and the two distinct Taylor allowances
For this single contact, Theorem 1's cross-contact assumption is vacuous:
$$f_0-f_\kappa=E_\kappa w,\quad w\in[1,2],\quad
E_\kappa=\frac{\tau}{D}[1,-\alpha/(1-\alpha),0]^T.$$
This disturbance is one-sided. We deliberately enclose it in a **zero-centered box**,
without shifting the prediction or incorrectly assuming $w\in[-1,1]$:
$$|f_0-f_\kappa|\le2|E_\kappa|\le
\boxed{B_{\rm smooth}=\sqrt\tau
[\sqrt{1-\alpha_{\rm lo}},\ \alpha_{\rm hi}/\sqrt{1-\alpha_{\rm hi}},\ 0]^T.}$$
The last inequality is uniform in excitation and across the entire mass prior. Thus the
repository's state-only `disturbance(X)` API is sufficient: no unimplemented input-dependent
disturbance adapter is hidden here. This conservative constant envelope loses local tightness.

For the estimator, the residual is
$$z^+_{\rm obs}-f_\kappa(z,U;\hat\theta)
=E_\theta(\theta_* -\hat\theta)+r_\theta+d_{\rm smooth}+d_{\rm num}+d_{\rm extra}.$$
We bound $r_\theta$ by a uniform second-derivative bound times
$\tfrac12(2\alpha_r)^2$ (both parameter centers and truth are in the prior).
The common generator used by the estimator **and information reward** is
$$F=\operatorname{diag}(B_{\rm smooth}+B_{\rm num}+B_{\rm extra}
 +B_{\rm float}+B_{\theta\text{-Taylor}}).$$
It is a conservative allowance, not noise that we inject into Drake.

The planner must also cover deviations in object position and command. On each tube box,
let $r_a=r_U+r_O$ and $r_\alpha=\alpha_r r_\theta$. The analytic remainder callback returns
$$R_i\le\tfrac12 M_{aa,i}r_a^2+M_{a\alpha,i}r_a r_\alpha
 +\tfrac12 M_{\alpha\alpha,i}r_\alpha^2.$$
`hessian_bounds` derives these bounds from $g=(a+D)/2$, $b=1-\alpha$:
$$g_a\in[0,1],\quad g_{aa}=2\tau/(bD^3),\quad
g_{a\alpha}=-\tau a/(b^2D^3),\quad
g_\alpha=\tau/(b^2D),\quad
g_{\alpha\alpha}=2\tau/(b^3D)-2\tau^2/(b^4D^3).$$
The parameter Taylor component is deliberately counted conservatively in both $F$ and the
full tube remainder. This avoids overstating the information reward while retaining the full
planner remainder. The small duplicated term can be tightened later.


In [ ]:
B_SMOOTH = np.sqrt(TAU)*np.array([np.sqrt(1-ALPHA_LO),ALPHA_HI/np.sqrt(1-ALPHA_HI),0.0])

def hessian_bounds(a_center,a_radius,alpha_lower,alpha_upper):
    # Valid on the whole (a,alpha) rectangle when alpha_upper < 1.
    b_min=1-alpha_upper
    b_max=1-alpha_lower
    valid=(b_min>0)&(b_max>0)
    b_min=jnp.maximum(b_min,1e-12)
    b_max=jnp.maximum(b_max,1e-12)
    a_min_abs=jnp.maximum(jnp.abs(a_center)-a_radius,0.0)
    D_min=jnp.sqrt(a_min_abs*a_min_abs+4*TAU/b_max)
    alpha_abs=jnp.maximum(jnp.abs(alpha_lower),jnp.abs(alpha_upper))
    # |a|/D <= 1. Bounds below need no gridding or numerical maximization.
    g_aa=2*TAU/(b_min*D_min**3)
    g_a_alpha=TAU/(b_min**2*D_min**2)
    g_alpha=TAU/(b_min**2*D_min)
    g_alpha_alpha=2*TAU/(b_min**3*D_min)+2*TAU**2/(b_min**4*D_min**3)
    M_aa=jnp.array([2*TAU/D_min**3,alpha_abs*g_aa,0.0])
    M_a_alpha=jnp.array([1+TAU/(b_min*D_min**2),1+alpha_abs*g_a_alpha,0.0])
    # For b*g, the second alpha derivative simplifies to -2*tau^2/(b^3*D^3).
    M_alpha_alpha=jnp.array([2*TAU**2/(b_min**3*D_min**3),
                            2*g_alpha+alpha_abs*g_alpha_alpha,0.0])
    return tuple(jnp.where(valid,v,jnp.full_like(v,jnp.inf)) for v in (M_aa,M_a_alpha,M_alpha_alpha))

_,_,M_ALPHA_GLOBAL=hessian_bounds(0.0,0.0,ALPHA_LO,ALPHA_HI)
B_PARAMETER_TAYLOR=np.asarray(0.5*M_ALPHA_GLOBAL*(2*ALPHA_R)**2)
B_NUMERICAL=DRAKE_NUMERICAL_BUDGET_M/LENGTH
B_EXTRA=EXTRA_PROCESS_BOUND_M/LENGTH
B_FLOAT=FLOAT_ALLOWANCE_M/LENGTH
B_PHYSICAL=B_SMOOTH+B_NUMERICAL+B_EXTRA+B_FLOAT
B_ESTIMATOR=B_PHYSICAL+B_PARAMETER_TAYLOR
F_MODEL=jnp.diag(jnp.asarray(B_ESTIMATOR))

def disturbance(X):
    return jnp.broadcast_to(F_MODEL,(X.shape[0],NX,NX))

def analytic_remainder(X,U,tube_x,tube_u,G_0,nominal_param):
    del G_0,nominal_param
    U_pad=jnp.concatenate([U,U[-1:]],axis=0)
    def one(s,u,rx,ru):
        a=u[0]-s[1]
        ra=jnp.abs(ru[0])+jnp.abs(rx[1])
        alpha=response_alpha(s[NX:])
        ralpha=ALPHA_R*jnp.abs(rx[NX])
        Maa,MaA,MAA=hessian_bounds(a,ra,alpha-ralpha,alpha+ralpha)
        bound=0.5*Maa*ra**2+MaA*ra*ralpha+0.5*MAA*ralpha**2
        return jnp.concatenate([bound,jnp.zeros(NT)])
    return jax.vmap(one)(X,U_pad,tube_x,tube_u)

for name,b in [('Smoothing theorem',B_SMOOTH),('Numerical budget',B_NUMERICAL),
               ('Extra process',B_EXTRA),('Parameter Taylor',B_PARAMETER_TAYLOR),
               ('Common estimator/information allowance',B_ESTIMATOR)]:
    print(name,'[pusher, object, previous command] um:',b*LENGTH*1e6)
print('State/input tube Taylor bounds are recomputed inside the adaptive solve.')


## Cell 5 — Drake observation bridge (separate environment)
The worker imports your **custom** `pydrake.all.QuasidynamicLinearPusher` and executes its
periodic discrete update. `DrakePlant.step` always requests `kappa=0`.
For optional source-parity audits only, the same worker can evaluate `kappa=KAPPA`.

Positions cross the bridge in meters. The JAX planner uses scaled coordinates. The worker
owns the hidden mass; that mass is never passed to `build_controller`, `solve_plan`, or
`estimator_update`. The analytical nonsmooth map audits numerical error after execution.
Only `EXTRA_PROCESS_BOUND_M` is optionally injected, separately from the genuine Drake result.


In [ ]:
import threading, queue

DRAKE_WORKER_SOURCE = r'''
import sys, json, traceback
import numpy as np
import pydrake
from pydrake.all import QuasidynamicLinearPusher
CACHE={}
PREFIX='NB06_JSON:'
for line in sys.stdin:
    try:
        request=json.loads(line)
        if request['op']=='close':
            break
        if request['op']=='info':
            response=dict(binding=pydrake.__file__,python=sys.executable,
                          system='QuasidynamicLinearPusher')
        else:
            key=tuple(request[k] for k in ('mass','kappa','ka','width','dt'))
            if key not in CACHE:
                system=QuasidynamicLinearPusher(k_a=key[2],m_o=key[0],w=key[3],dt=key[4],dynamics_smoothing=key[1])
                CACHE[key]=(system,system.CreateDefaultContext())
            system,context=CACHE[key]
            context.SetDiscreteState(np.asarray(request['state'],dtype=float))
            system.get_input_port().FixValue(context,np.array([request['command']],dtype=float))
            update=system.EvalUniquePeriodicDiscreteUpdate(context)
            E=None
            if hasattr(system,'GetDisturbanceMatrix'):
                E=np.asarray(system.GetDisturbanceMatrix(context)).tolist()
            context.SetDiscreteState(update)
            response=dict(next_state=context.get_discrete_state_vector().CopyToVector().tolist(),E=E)
        print(PREFIX+json.dumps(dict(ok=True,result=response)),flush=True)
    except Exception:
        print(PREFIX+json.dumps(dict(ok=False,error=traceback.format_exc())),flush=True)
'''

class DrakeBridge:
    def __init__(self):
        self.directory=Path(tempfile.mkdtemp(prefix='nb06_drake_'))
        worker=self.directory/'worker.py'
        worker.write_text(DRAKE_WORKER_SOURCE)
        self.error_path=self.directory/'stderr.log'
        self.error_stream=self.error_path.open('w')
        self.messages=queue.Queue()
        self.logs=[]
        self.process=subprocess.Popen(DRAKE_LAUNCH+[str(worker)],stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,stderr=self.error_stream,text=True,bufsize=1)
        def reader():
            try:
                for line in self.process.stdout:
                    if line.startswith('NB06_JSON:'):
                        self.messages.put(json.loads(line[len('NB06_JSON:'):]))
                    else:
                        self.logs.append(line.rstrip())
            finally:
                self.messages.put(dict(ok=False,error='Drake worker closed its stdout.'))
        self.thread=threading.Thread(target=reader,daemon=True)
        self.thread.start()
        try:
            self.info=self.call(dict(op='info'))
        except Exception:
            self.close()
            raise

    def call(self,request):
        try:
            self.process.stdin.write(json.dumps(request)+'\n')
            self.process.stdin.flush()
            response=self.messages.get(timeout=RPC_TIMEOUT_SECONDS)
        except (queue.Empty,BrokenPipeError) as exc:
            tail=self.error_path.read_text()[-6000:]
            self.close()
            raise RuntimeError('Drake bridge failed/timed out. Check DRAKE_LAUNCH and custom binding.\n'+tail) from exc
        if not response['ok']:
            tail=self.error_path.read_text()[-6000:]
            raise RuntimeError(response.get('error','Drake worker failed')+'\n'+tail)
        return response['result']

    def transition(self,z,u,mass,kappa):
        absolute_state=[float(z[0])*LENGTH+O_ORIGIN-WIDTH,
                        float(z[1])*LENGTH+O_ORIGIN]
        result=self.call(dict(op='step',mass=float(mass),kappa=float(kappa),ka=KA,width=WIDTH,
            dt=DT,state=absolute_state,command=float(u[0])*LENGTH+O_ORIGIN-WIDTH))
        p,o=result['next_state']
        next_z=np.array([(p-(O_ORIGIN-WIDTH))/LENGTH,(o-O_ORIGIN)/LENGTH,float(u[0])])
        return next_z,result

    def close(self):
        process=getattr(self,'process',None)
        if process is not None and process.poll() is None:
            try:
                process.stdin.write('{"op":"close"}\n'); process.stdin.flush()
                process.wait(timeout=2)
            except Exception:
                process.terminate()
        stream=getattr(self,'error_stream',None)
        if stream is not None:
            stream.close()

class DrakePlant:
    def __init__(self,bridge,hidden_mass):
        if not MASS_PRIOR[0]<=hidden_mass<=MASS_PRIOR[1]:
            raise ValueError('Hidden evaluation mass is outside the stated prior.')
        self.bridge=bridge
        self.hidden_mass=float(hidden_mass)
        self.hidden_theta=float((alpha_from_mass(hidden_mass)-ALPHA_C)/ALPHA_R)

    def step(self,z,u,xi):
        xi=np.asarray(xi,dtype=float)
        if xi.shape!=(NX,) or np.max(np.abs(xi))>1+1e-12:
            raise ValueError('Invalid optional process-disturbance sample.')
        drake_z,raw=self.bridge.transition(z,u,self.hidden_mass,kappa=0.0)
        ideal=np.asarray(exact_nonsmooth_step(z,u,jnp.array([self.hidden_theta])))
        smooth=np.asarray(physical_step(z,u,jnp.array([self.hidden_theta])))
        numerical_error_m=(drake_z-ideal)*LENGTH
        smoothing_error_m=(ideal-smooth)*LENGTH
        next_z=drake_z+(EXTRA_PROCESS_BOUND_M/LENGTH)*xi
        audit=dict(numerical_error_m=numerical_error_m,smoothing_error_m=smoothing_error_m,
            numerical_ok=bool(np.all(np.abs(numerical_error_m)<=DRAKE_NUMERICAL_BUDGET_M+FLOAT_ALLOWANCE_M)),
            smoothing_ok=bool(np.all(np.abs(smoothing_error_m)<=B_SMOOTH*LENGTH+FLOAT_ALLOWANCE_M)),
            actual_smooth_residual_m=(next_z-smooth)*LENGTH,
            drake_next=drake_z,raw=raw)
        return jnp.asarray(next_z),audit

print('Drake worker and nonsmooth execution adapter defined. No worker has started yet.')


## Cell 6 — Disturbance cost, linear constraints, and seed trajectory
The base cost is the earlier object-displacement, force, and command-change penalty.
The adaptive repository adds its information reward separately in Cell 9:
$$J=J_d-\lambda\sum_{k=0}^{N-1}
\operatorname{tr}(G_kG_k^T-G_{k+1}G_{k+1}^T).$$
The repository evaluates this using lagged generators; it is not the fully coupled exact
experiment objective. Here its stage noise map includes smoothing and parameter-Taylor allowances.

We impose $a\ge a_{\min}>0$. The smooth contact force decreases with $\alpha$, so its
upper envelope uses $\alpha_{\rm lo}$. If $T=F_{\rm cap}/(K_aL)$, inversion of
$(1-\alpha_{\rm lo})g_\kappa(a,\alpha_{\rm lo})\le T$ gives
$$a\le T/(1-\alpha_{\rm lo})-\tau/T.$$
The force budget also reserves numerical/extra-process error in pusher position.

**Important API detail:** the first three rows are upper bounds for $P,O,R$, then their
lower bounds, then coupled command constraints, and finally input upper/lower bounds.
`gpu_sls.py` reads tube widths from these positions. The old arbitrary eight-row layout
cannot safely be reused with this remainder callback.


In [ ]:
FORCE_NOISE_MARGIN=KA*(DRAKE_NUMERICAL_BUDGET_M[0]+EXTRA_PROCESS_BOUND_M[0]+FLOAT_ALLOWANCE_M[0])
T_FORCE=(FORCE_MAX-FORCE_NOISE_MARGIN)/(KA*LENGTH)
assert T_FORCE>0
A_FORCE_CAP=T_FORCE/(1-ALPHA_LO)-TAU/T_FORCE
assert A_FORCE_CAP>A_MIN
Z_LO=jnp.array([U_MIN-FORCE_MAX/(KA*LENGTH)-0.1,-BACKWARD_ALLOWANCE,U_MIN])
Z_HI=jnp.array([U_MAX,MOVE_MAX,U_MAX])

def base_cost(W,reference,s,u,t):
    displacement=W[0]*((s[1]-reference[t,1])/MOVE_SCALE)**2
    force_proxy=KA*LENGTH*(1-ALPHA_LO)*barrier_excitation(u[0]-s[1],ALPHA_LO)
    stage=displacement+W[1]*(force_proxy/FORCE_SCALE)**2+W[2]*((u[0]-s[2])/COMMAND_SCALE)**2
    stage+=1e-6*(jnp.dot(s,s)+jnp.dot(u,u))
    terminal=displacement+1e-6*jnp.dot(s,s)
    return jnp.where(t<HORIZON,stage/HORIZON,terminal/HORIZON)

CONSTRAINT_NAMES=['pusher upper','object upper','previous-command upper',
                  'pusher lower','object lower','previous-command lower',
                  'command increment','command decrement','minimum excitation',
                  'force-equivalent excitation cap','input upper','input lower']
NC=len(CONSTRAINT_NAMES)

def constraints(s,u,t):
    a=u[0]-s[1]; du=u[0]-s[2]
    boxes=jnp.concatenate([s[:NX]-Z_HI,Z_LO-s[:NX]])
    controls=jnp.array([du-DU_MAX,-du-DU_MAX,A_MIN-a,a-A_FORCE_CAP,u[0]-U_MAX,U_MIN-u[0]])
    controls=jnp.where(t<HORIZON,controls,-jnp.ones_like(controls))
    return jnp.concatenate([boxes,controls])

def nominal_rollout(s0,U):
    def step(s,pair):
        k,u=pair
        nxt=dynamics(s,u,k,parameter=DT)
        return nxt,nxt
    _,following=jax.lax.scan(step,s0,(jnp.arange(U.shape[0]),U))
    return jnp.concatenate([s0[None,:],following],axis=0)

def make_seed(s0,excitation=3e-3/LENGTH):
    s=s0; commands=[]
    for k in range(HORIZON):
        u=jnp.array([s[1]+excitation]); commands.append(u)
        s=dynamics(s,u,k,parameter=DT)
    U=jnp.stack(commands)
    return nominal_rollout(s0,U),U

def stage_residuals(X,U):
    U_pad=jnp.concatenate([U,jnp.zeros((1,NU))],axis=0)
    return np.asarray(jax.vmap(constraints)(X,U_pad,jnp.arange(HORIZON+1)))

print('Constraint convention <= 0; rows:',NC)
print('Force-equivalent maximum excitation [mm]:',A_FORCE_CAP*LENGTH*1e3)


## Cell 7 — Repository estimator with a valid nonlinear-parameter allowance
The current transition uses the observed state and selected input:
$$E_\theta=\partial f_\kappa/\partial\theta,\quad
r=z^+_{\rm obs}-f_\kappa(z,U;\hat\theta),\quad
\hat\theta^+=\hat\theta+L P_{\rm obs}r.$$
`observable_subspace_factorizer` maps the object-position residual and the **common error
generator from Cell 4** into the repository's observable measurement coordinates.
`adaptive_posterior_factors` and `girard_reduction_opt` implement its update.

We then explicitly intersect the scalar posterior with the original physical prior and
recenter/rescale its generators. This is a new, exact **one-dimensional** set operation.
It keeps the next expansion center in the domain where the parameter-Taylor bound applies.
It differs from the old notebook's reporting-only clipping and is logged.

A separate nonsmooth interval estimator is a diagnostic only. It uses the exact object
equation and the numerical/extra-process budgets, not the smoothing allowance, and is
**not** fed into the adaptive solver. No true mass is used by either estimator.


In [ ]:
@jax.jit
def repository_estimator_update(s_previous,u,observed_next,G):
    theta=s_previous[NX:]
    predictor=lambda candidate:physical_step(s_previous[:NX],u,candidate)
    prediction=predictor(theta)
    E_theta=jax.jacfwd(predictor)(theta)
    P_obs,Pi,W_obs=observable_subspace_factorizer(E_theta,F_MODEL,C_MEAS)
    _,Lgain,G_pre,_=adaptive_posterior_factors(G,Pi,W_obs,
        gain_regularization=GAIN_REG,enable_posterior_gate=True)
    residual=observed_next-prediction
    innovation=P_obs@residual
    theta_next=theta+Lgain@innovation
    G_next=girard_reduction_opt(G_pre,QMAX)
    return jnp.concatenate([observed_next,theta_next]),G_next,dict(
        E=E_theta,W=W_obs,L=Lgain,residual=residual,innovation=innovation,Pi=Pi)

def estimator_update(s_previous,u,observed_next,G):
    updated,G_next,audit=repository_estimator_update(s_previous,u,observed_next,G)
    raw_lo,raw_hi=parameter_interval(updated[NX:],G_next)
    lo,hi=max(raw_lo,-1.0),min(raw_hi,1.0)
    if lo>hi:
        raise RuntimeError('Posterior has empty intersection with the physical prior.')
    changed=(lo!=raw_lo or hi!=raw_hi)
    if changed:
        radius=(hi-lo)/2
        raw_radius=(raw_hi-raw_lo)/2
        G_next=G_next*(radius/raw_radius) if raw_radius>0 else jnp.zeros_like(G_next)
        updated=updated.at[NX].set((lo+hi)/2)
    audit=dict(audit,prior_intersection_applied=changed,raw_interval=np.array([raw_lo,raw_hi]))
    return updated,G_next,audit

def independent_interval_update(interval,z,u,next_z):
    a=max(float(u[0]-z[1]),0.0)
    H=ALPHA_R*a
    if H<=1e-12:
        return interval.copy()
    r=float(next_z[1]-z[1])-ALPHA_C*a
    b=float(B_NUMERICAL[1]+B_EXTRA[1]+B_FLOAT[1])
    result=np.array([max(interval[0],(r-b)/H),min(interval[1],(r+b)/H)])
    if result[0]>result[1]:
        raise RuntimeError('Empty independent nonsmooth interval: model/budget inconsistency.')
    return result

print('Adaptive repository estimator plus nonlinear-parameter allowance and scalar prior intersection defined.')


## Cell 8 — Optional manual source/interface audits
These checks are supplied for your run; they were **not run when this notebook was built**.
They compare the JAX smooth solution with the actual smooth C++ transition, compare the
theorem generator when its binding is available, and test nonsmooth numerical acceptance
on active-contact examples. Finite examples are not a global numerical-error proof.

`BRIDGE` starts here and is reused by subsequent experiments. If you skip the audit,
the worker still starts so the later execution layer is available.


In [ ]:
if 'BRIDGE' in globals():
    BRIDGE.close()
BRIDGE=DrakeBridge()
PROVENANCE['drake_runtime']=BRIDGE.info
print('Drake worker:',BRIDGE.info)
AUDIT_RESULTS=[]
if RUN_INTERFACE_AUDIT:
    smooth_tolerance_m=np.array([0.5e-6,0.5e-6,1e-12])
    for mass in [MASS_PRIOR[0],0.015,MASS_PRIOR[1]]:
        theta=jnp.array([(alpha_from_mass(mass)-ALPHA_C)/ALPHA_R])
        plant=DrakePlant(BRIDGE,mass)
        for excitation_m in [1e-3,5e-3,15e-3]:
            z=jnp.array([0.0,0.0,0.0]); u=jnp.array([excitation_m/LENGTH])
            smooth_cpp,raw=BRIDGE.transition(z,u,mass,KAPPA)
            smooth_jax=np.asarray(physical_step(z,u,theta))
            smooth_error_m=(smooth_cpp-smooth_jax)*LENGTH
            if np.any(np.abs(smooth_error_m)>smooth_tolerance_m):
                raise RuntimeError(f'Smooth C++/JAX mismatch: mass={mass}, error_m={smooth_error_m}')
            E_error=None
            if raw['E'] is not None:
                E_cpp=np.asarray(raw['E']).reshape(2)/LENGTH
                E_expected=np.asarray(weichen_E(z,u,theta))[:2]
                E_error=float(np.max(np.abs(E_cpp-E_expected)))
                if not np.allclose(E_cpp,E_expected,rtol=1e-4,atol=1e-8):
                    raise RuntimeError('Drake smoothing generator does not match the specialized formula.')
            next_z,audit=plant.step(z,u,np.zeros(NX))
            if not (audit['numerical_ok'] and audit['smoothing_ok']):
                raise RuntimeError(f'Drake/theorem audit failed: {audit}')
            s0=jnp.concatenate([z,INITIAL_THETA])
            s1,G1,_=estimator_update(s0,u,next_z,INITIAL_G)
            lo,hi=parameter_interval(s1[NX:],G1)
            if not lo-1e-9<=float(theta[0])<=hi+1e-9:
                raise RuntimeError('Fixture estimator lost the true parameter.')
            AUDIT_RESULTS.append(dict(mass_g=mass*1e3,excitation_mm=excitation_m*1e3,
                smooth_error_um=(smooth_error_m*1e6).tolist(),
                numerical_error_um=(audit['numerical_error_m']*1e6).tolist(),E_error=E_error))
    print('Completed source/interface fixtures:',len(AUDIT_RESULTS))
    print('These finite fixtures do not prove global numerical accuracy or controller convergence.')
else:
    print('Source/interface audits skipped by user setting.')


## Cell 9 — Complete adaptive controller with the analytic remainder callback
`GenericMPCControllerWrapper` supplies the controller state and receding-horizon update.
Its `_solve` callable is bound to the **same repository `sqp.mpc`** with our analytic
remainder callback replacing its generic interval-Hessian callback. No repository file is
changed, and no custom outer/inner trajectory optimizer is introduced.

The solver receives smooth dynamics, constraints, $J_d$, the information weight, the common
error generator, the current parameter generators, and the analytic Taylor bound. It returns
the first command, full nominal plan, tightenings, predicted generators, and diagnostics.

Both SLS and ADMM convergence, finite values, nominal dynamics consistency, and constraint
checks are required before execution. These are numerical acceptance gates, not a claim of
global optimality. If the solve fails with the bounds enabled, diagnose it; do not silently
disable the bounds or execute a fallback plan.


In [ ]:
@dataclass
class PusherConfig:
    n: int=NA
    nu: int=NU
    N: int=HORIZON
    W: object=None
    u_ref: object=None
    dt: float=DT

DIAGNOSTIC_NAMES=['sls_converged','sls_residual','admm_converged','admm_iterations',
    'admm_primal_residual','admm_dual_residual','admm_primal_tolerance','admm_dual_tolerance',
    'admm_primal_worst_flat_index','admm_primal_worst_z','admm_primal_worst_w']

def build_controller(info_weight=None):
    if info_weight is None:
        info_weight=INFO_WEIGHT
    start=jnp.concatenate([INITIAL_PHYS,INITIAL_THETA])
    X,U=make_seed(start)
    cfg=PusherConfig(W=W_COST,u_ref=jnp.array([0.0]))
    sls=SLSConfig(max_sls_iterations=20,sls_primal_tol=1e-5,enable_fastsls=True,
        warm_start=False,n_x=NX,n_w=NX,n_theta=NT,num_param=NT,q_max=QMAX,adaptive=True,
        enable_linearization_error=True,enable_disturbance_variation_error=False,
        enable_posterior_gate=True,enable_information_cost=info_weight>0,
        information_cost_weight=float(info_weight),information_cost_discount=1.0,
        information_gain_regularization=GAIN_REG)
    sqp=SQPConfig(max_sqp_iterations=20,warm_start=False,feas_tol=1e-6,step_tol=1e-6,line_search=True)
    admm=ADMMConfig(max_iterations=1500,eps_abs=1e-6,eps_rel=1e-5,initial_rho=10.0,
                    rho_max=1e6,rho_update_frequency=25)
    controller=GenericMPCControllerWrapper(sls,sqp,admm,config=cfg,
        dynamics=dynamics,constraints=constraints,obstacles=OBSTACLES,
        cost=base_cost,num_constraints=NC,disturbance=disturbance,X_in=X,U_in=U,
        limited_memory=False,shift=1,measurement_matrix=C_MEAS,
        Q_bar=jnp.diag(jnp.array([1e-3,1.0,1e-3,1e-3])),R_bar=jnp.array([[0.1]]),
        Q_f_bar=jnp.diag(jnp.array([1e-3,1.0,1e-3,1e-3])))
    # The wrapper's current public constructor lacks a custom-remainder argument.
    # Rebind only this instance to the same solver with our explicit bound callback.
    work=partial(sqp_module.mpc,sls,sqp,admm,base_cost,dynamics,None,False,
        constraints,disturbance,analytic_remainder,(4,4,4,4),C_MEAS,
        Q_bar=controller.Q_bar,R_bar=controller.R_bar,Q_f_bar=controller.Q_f_bar)
    compiled=jax.jit(work)
    def recorded_solve(*args,**kwargs):
        result=compiled(*args,**kwargs)
        if len(result)!=28:
            raise RuntimeError('Expected current 28-value solver return. Update/inspect the adaptive checkout; no silent layout fallback.')
        controller.notebook_diagnostics=dict(zip(DIAGNOSTIC_NAMES,result[17:]))
        return result
    controller._solve=recorded_solve
    controller.notebook_diagnostics={}
    controller.notebook_info_weight=float(info_weight)
    return controller

def solve_plan(controller,s,G):
    reference=jnp.zeros((HORIZON+1,NA)).at[:,NX:].set(s[NX:])
    controller.X0=nominal_rollout(s,controller.U0)
    start=time.perf_counter()
    out=controller.run(x0=s,G_0=G,reference=reference,nominal_param=s[NX:],parameter=DT)
    out[0].block_until_ready()
    u,X,U,V,backoffs,Phi_x,Phi_u,K,EN,KN,GN,P_inv=out
    diag={k:np.asarray(v).item() for k,v in controller.notebook_diagnostics.items()}
    if not all(np.isfinite(np.asarray(a)).all() for a in [u,X,U,backoffs,GN]):
        raise FloatingPointError(f'Nonfinite solver output; inspect domain/bounds/solver status: {diag}')
    rollout=nominal_rollout(s,U)
    defects=np.asarray(X-rollout)
    residuals=stage_residuals(X,U)+np.asarray(backoffs)
    worst=np.unravel_index(np.argmax(residuals),residuals.shape)
    radius=float(np.abs(np.asarray(G)).sum())
    centers=np.asarray(X)[:,NX]
    # This notebook's parameter-only allowance was derived over the physical prior.
    parameter_domain_ok=bool(np.min(centers-radius)>=-1-1e-8 and np.max(centers+radius)<=1+1e-8)
    diag.update(seconds=time.perf_counter()-start,dynamics_defect=float(np.max(np.abs(defects))),
        nominal_violation=float(np.max(stage_residuals(rollout,U))),
        tightened_violation=float(np.max(residuals)),parameter_domain_ok=parameter_domain_ok,
        worst_constraint=CONSTRAINT_NAMES[worst[1]],worst_stage=int(worst[0]))
    # Record the actual remainder callback evaluated on the returned widths as well.
    tx=np.concatenate([np.asarray(backoffs)[:,:NX],np.full((HORIZON+1,NT),radius)],axis=1)
    tu=np.asarray(backoffs)[:,NC-2*NU:NC-NU]
    remainder=np.asarray(analytic_remainder(X,U,jnp.asarray(tx),jnp.asarray(tu),G,s[NX:]))
    diag['max_taylor_allowance_um']=(np.max(remainder[:,:NX],axis=0)*LENGTH*1e6).tolist()
    return dict(u=u,X=X,U=U,backoffs=backoffs,GN=GN,diagnostics=diag,remainder=remainder,
                info_weight=controller.notebook_info_weight)

def plan_is_acceptable(plan):
    d=plan['diagnostics']
    return bool(d['admm_converged'] and d['sls_converged'] and d['parameter_domain_ok']
                and d['dynamics_defect']<=DYNAMICS_DEFECT_TOL
                and d['nominal_violation']<=CHECK_TOL
                and d['tightened_violation']<=CHECK_TOL
                and np.isfinite(plan['remainder']).all())

print('Controller uses repository SQP/SLS/ADMM, information reward, and our analytic tube remainder.')


## Cell 10 — Independent one-step bound and a demonstration plan
This guard uses the **current parameter interval**, not the hidden mass. Smooth pusher and
object positions are monotone in $\alpha$ on the physical prior, so endpoint predictions
bound them. Apply the theorem's known signs: nonsmooth pusher position is above the smooth
one; nonsmooth object position is below it. Add numerical and optional process budgets.

This cell solves one horizon and checks the first command. It **does not execute** any
command or update an estimate. Cell 11 starts independently from the initial state.


In [ ]:
def first_step_guard(s,G,u):
    lo,hi=parameter_interval(s[NX:],G)
    lo,hi=max(lo,-1.0),min(hi,1.0)
    if lo>hi:
        return False,dict(reason='empty parameter/prior intersection')
    lower=np.asarray(physical_step(s[:NX],u,jnp.array([lo])))
    upper=np.asarray(physical_step(s[:NX],u,jnp.array([hi])))
    extra=B_NUMERICAL+B_EXTRA+B_FLOAT
    z_lo=lower-np.array([0.0,B_SMOOTH[1],0.0])-extra
    z_hi=upper+np.array([B_SMOOTH[0],0.0,0.0])+extra
    force_hi=KA*LENGTH*(float(u[0])-z_lo[0])
    a=float(u[0]-s[1]); du=float(u[0]-s[2])
    residuals=np.concatenate([z_hi-np.asarray(Z_HI),np.asarray(Z_LO)-z_lo,
        np.array([float(u[0])-U_MAX,U_MIN-float(u[0]),abs(du)-DU_MAX,
                  A_MIN-a,a-A_FORCE_CAP,(force_hi-FORCE_MAX)/(KA*LENGTH)])])
    return bool(np.isfinite(residuals).all() and np.max(residuals)<=CHECK_TOL),dict(
        max_residual=float(np.max(residuals)),force_upper_N=force_hi,
        object_lower_m=float(z_lo[1]*LENGTH+O_ORIGIN),object_upper_m=float(z_hi[1]*LENGTH+O_ORIGIN),
        next_state_lower_scaled=z_lo,next_state_upper_scaled=z_hi)

DEMO_PLAN=None
if RUN_DEMO:
    controller_demo=build_controller(info_weight=INFO_WEIGHT)
    s0=jnp.concatenate([INITIAL_PHYS,INITIAL_THETA])
    print('Compiling and solving the smooth-model plan with error bounds enabled...',flush=True)
    DEMO_PLAN=solve_plan(controller_demo,s0,INITIAL_G)
    ok,guard=first_step_guard(s0,INITIAL_G,DEMO_PLAN['u'])
    print('Information weight:',DEMO_PLAN['info_weight'])
    print('Diagnostics:',DEMO_PLAN['diagnostics'])
    print('Plan accepted:',plan_is_acceptable(DEMO_PLAN),'| first-step guard:',ok,guard)
    print('Six command offsets [mm]:',np.asarray(DEMO_PLAN['U']).ravel()*LENGTH*1e3)
    print('Nominal object displacement [mm]:',np.asarray(DEMO_PLAN['X'])[:,1]*LENGTH*1e3)
    print('No command executed in this cell.')


## Cell 11 — Closed-loop plan → Drake execution → observation → learning
Each experiment builds a fresh controller. Only the execution/evaluation layer receives
`hidden_mass`. At each step:
1. Plan all six commands using the current observation, parameter estimate and generators.
2. Require solver/constraint acceptance and the independent first-step guard.
3. Execute just the first command in **nonsmooth Drake**.
4. Audit numerical/smoothing bounds before trusting that observation for learning.
5. Update the actual repository estimator; keep the independent interval as a diagnostic.
6. Replan from the resulting observed state. The old five commands are a shifted seed,
   not an unchangeable execution queue.

Failure stops the experiment with its diagnostics. No bound is automatically enlarged,
no smoothing error is replaced by random noise, and no analytical plant fallback is used.


In [ ]:
def run_experiment(hidden_mass=0.015,seed=7,info_weight=None,steps=None):
    if info_weight is None:
        info_weight=INFO_WEIGHT
    if steps is None:
        steps=EXECUTION_STEPS
    plant=DrakePlant(BRIDGE,hidden_mass)
    controller=build_controller(info_weight=float(info_weight))
    noise=np.random.default_rng(seed).uniform(-1,1,(steps,NX)); noise[:,2]=0.0
    s=jnp.concatenate([INITIAL_PHYS,INITIAL_THETA]); G=INITIAL_G
    independent=np.array([-1.0,1.0])
    states=[np.asarray(s)]; generators=[np.asarray(G)]; intervals=[reported_mass_interval(s[NX:],G)]
    rows=[]; plans=[]; execution_audits=[]; estimator_audits=[]
    status='completed'; failure=None
    for k in range(steps):
        print(f'Planning step {k+1}/{steps}...',flush=True)
        try:
            plan=solve_plan(controller,s,G)
        except Exception as exc:
            status='stopped_before_execution: solver exception'
            failure=repr(exc); print(status,failure); break
        plans.append(plan)
        guard_ok,guard=first_step_guard(s,G,plan['u'])
        diag=plan['diagnostics']
        print(f"  solve={diag['seconds']:.2f}s, ADMM={diag['admm_converged']}, SLS={diag['sls_converged']}, guard={guard_ok}",flush=True)
        if not (plan_is_acceptable(plan) and guard_ok):
            status='stopped_before_execution: plan or guard rejected'
            failure=dict(diagnostics=diag,guard=guard)
            print(status,failure); break
        old_s=s; u=plan['u']
        try:
            next_z,audit=plant.step(old_s[:NX],u,noise[k])
        except Exception as exc:
            status='stopped_during_execution: Drake bridge exception'
            failure=repr(exc); print(status,failure); break
        execution_audits.append(dict(audit,observed_next=np.asarray(next_z),step=k+1))
        if not (audit['numerical_ok'] and audit['smoothing_ok']):
            status='stopped_after_execution: numerical or smoothing bound exceeded'
            failure=audit; print(status,audit); break
        try:
            independent=independent_interval_update(independent,old_s[:NX],u,next_z)
            s,G,est=estimator_update(old_s,u,next_z,G)
        except Exception as exc:
            status='stopped_after_execution: estimator inconsistency'
            failure=repr(exc); print(status,failure); break
        estimator_audits.append(est)
        mi=reported_mass_interval(s[NX:],G)
        lo,hi=parameter_interval(s[NX:],G)
        covered=bool(lo-1e-9<=plant.hidden_theta<=hi+1e-9)
        # Truth is used only for this audit, never as an estimator/solver input.
        affine_residual=np.asarray(est['residual']-est['E'][:,0]*(plant.hidden_theta-float(old_s[NX])))
        residual_bound_ok=bool(np.all(np.abs(affine_residual)<=B_ESTIMATOR+1e-10))
        force=KA*LENGTH*(float(u[0])-float(next_z[0]))
        motion=float(next_z[1])*LENGTH
        actual_limits=bool(np.max(np.asarray(next_z)-np.asarray(Z_HI))<=CHECK_TOL
            and np.max(np.asarray(Z_LO)-np.asarray(next_z))<=CHECK_TOL
            and force<=FORCE_MAX+KA*LENGTH*CHECK_TOL)
        estimated_mass=float(mass_from_alpha(ALPHA_C+ALPHA_R*float(s[NX])))
        rows.append(dict(step=k+1,command_m=float(u[0])*LENGTH+O_ORIGIN-WIDTH,
            object_travel_mm=motion*1e3,force_mN=force*1e3,mass_estimate_g=estimated_mass*1e3,
            mass_lo_g=mi[0]*1e3,mass_hi_g=mi[1]*1e3,width_g=(mi[1]-mi[0])*1e3,
            covered=covered,bound_ok=residual_bound_ok,limits_ok=actual_limits,
            solve_seconds=diag['seconds'],independent_theta_width=float(independent[1]-independent[0]),
            posterior_radius=float(np.abs(np.asarray(G)).sum()),
            numerical_pusher_error_um=float(audit['numerical_error_m'][0]*1e6),
            numerical_object_error_um=float(audit['numerical_error_m'][1]*1e6),
            smoothing_object_error_um=float(audit['smoothing_error_m'][1]*1e6),
            prior_intersection=bool(est['prior_intersection_applied']),
            admm_converged=bool(diag['admm_converged']),sls_converged=bool(diag['sls_converged'])))
        states.append(np.asarray(s)); generators.append(np.asarray(G)); intervals.append(mi)
        print(f'  Drake travel={motion*1e3:.4f} mm, mass interval={mi*1e3} g, covered={covered}',flush=True)
        if not (covered and residual_bound_ok and actual_limits):
            status='stopped_after_execution: coverage, residual bound, or physical limit failed'
            failure=rows[-1]; print(status,failure); break
    return dict(rows=rows,states=np.asarray(states),generators=np.asarray(generators),
        mass_intervals=np.asarray(intervals),plans=plans,execution_audits=execution_audits,
        estimator_audits=estimator_audits,status=status,failure=failure,
        hidden_mass=hidden_mass,seed=seed,info_weight=float(info_weight),requested_steps=steps,
        backend='Wei-Chen nonsmooth Drake',prediction='fixed-kappa log-barrier JAX port')

RESULT=run_experiment(info_weight=INFO_WEIGHT) if RUN_CLOSED_LOOP else None
if RESULT is not None:
    print('Status:',RESULT['status'],'| learned from',len(RESULT['rows']),'/',RESULT['requested_steps'],'executions')
else:
    print('Closed-loop execution disabled.')


## Cell 12 — Results and error-budget diagnostics
These plots come only from your run. The first two show task disturbance and mass learning.
The final plot separates the analytic smoothing discrepancy from numerical Drake error.
An interval that stops shrinking is a result to investigate, not evidence that later steps
were omitted. If the experiment stopped, inspect the recorded failure before interpreting it.


In [ ]:
if RESULT is not None:
    print('Status:',RESULT['status'])
    print('Recorded information weight:',RESULT['info_weight'])
    if RESULT['failure'] is not None:
        print('Failure:',RESULT['failure'])
    rows=RESULT['rows']
    if rows:
        for r in rows:
            print(f"step {r['step']}: travel={r['object_travel_mm']:.4f} mm, force={r['force_mN']:.3f} mN, width={r['width_g']:.4f} g, covered={r['covered']}, bound={r['bound_ok']}")
        states=RESULT['states']; masses=RESULT['mass_intervals']*1e3
        k=np.arange(len(states))
        fig,axes=plt.subplots(1,3,figsize=(17,4))
        axes[0].plot(k,states[:,1]*LENGTH*1e3,'o-',label='Nonsmooth Drake observation')
        axes[0].set(xlabel='Executed commands',ylabel='Object displacement [mm]')
        axes[0].legend()
        axes[1].fill_between(k,masses[:,0],masses[:,1],alpha=0.25,label='Adaptive mass interval')
        point=mass_from_alpha(ALPHA_C+ALPHA_R*states[:,NX])*1e3
        axes[1].plot(k,point,'o-',label='Estimate')
        axes[1].axhline(RESULT['hidden_mass']*1e3,ls='--',color='k',label='Hidden truth (evaluation only)')
        axes[1].set(xlabel='Executed commands',ylabel='Mass [g]'); axes[1].legend()
        kk=np.arange(1,len(rows)+1)
        axes[2].plot(kk,[abs(r['smoothing_object_error_um']) for r in rows],'o-',label='Actual |nonsmooth − smooth|')
        axes[2].plot(kk,[abs(r['numerical_object_error_um']) for r in rows],'s-',label='Drake numerical error')
        axes[2].axhline(B_SMOOTH[1]*LENGTH*1e6,ls='--',label='Theorem envelope')
        axes[2].axhline(DRAKE_NUMERICAL_BUDGET_M[1]*1e6,ls=':',label='Numerical budget')
        axes[2].set(xlabel='Executed commands',ylabel='Object error [um]'); axes[2].legend(fontsize=8)
        for ax in axes: ax.grid(alpha=0.3)
        fig.tight_layout(); plt.show()
        print('Final mass width [g]:',rows[-1]['width_g'])
        print('Final object displacement [mm]:',rows[-1]['object_travel_mm'])
        print('Peak executed model force [mN]:',max(r['force_mN'] for r in rows))
    else:
        print('No accepted observations for learning; no performance result can be claimed.')


## Cell 13 — Optional information-weight comparison
Each call resets the plant state, parameter prior, and controller. Both use the same
hidden mass, numerical acceptance budgets, physical limits, and optional process-noise seed.
Compare achieved uncertainty **and** displacement; information weight alone does not promise
that one plan dominates another. This remains a single simulated test unless you run a sweep.


In [ ]:
COMPARISON={}
if RUN_COMPARISON:
    for weight in [0.0,INFO_WEIGHT]:
        COMPARISON[weight]=run_experiment(info_weight=weight)
    for weight,result in COMPARISON.items():
        rows=result['rows']
        print('weight:',weight,'| status:',result['status'],'| completed:',len(rows))
        if rows:
            print('  displacement [mm]:',rows[-1]['object_travel_mm'],
                  '| interval width [g]:',rows[-1]['width_g'])
else:
    print('Comparison disabled. Set RUN_COMPARISON=True to run both fresh experiments.')


## Cell 14 — Save results and close the worker when finished
Exports are timestamped, include bounds/source hashes/settings, and do not overwrite earlier
results. Use the displayed status when reporting results; a stopped run is not a completed test.
Close the worker only after any reruns/comparisons are finished; rerun Cell 8 to restart it.


In [ ]:
SAVE_RESULTS=False
CLOSE_DRAKE_WORKER=False

def jsonable(value):
    if isinstance(value,dict): return {str(k):jsonable(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [jsonable(v) for v in value]
    if hasattr(value,'shape'): return np.asarray(value).tolist()
    if isinstance(value,Path): return str(value)
    return value

if SAVE_RESULTS and RESULT is not None:
    project=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
    destination=project/'results'/('nb06_drake_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
    destination.mkdir(parents=True,exist_ok=False)
    settings=dict(dt=DT,ka=KA,width=WIDTH,length=LENGTH,kappa=KAPPA,mass_prior=MASS_PRIOR,
        information_weight=RESULT['info_weight'],horizon=HORIZON,execution_steps=EXECUTION_STEPS,
        smoothing_bound_scaled=B_SMOOTH,numerical_budget_m=DRAKE_NUMERICAL_BUDGET_M,
        extra_process_bound_m=EXTRA_PROCESS_BOUND_M,float_allowance_m=FLOAT_ALLOWANCE_M,
        parameter_taylor_scaled=B_PARAMETER_TAYLOR,common_bound_scaled=B_ESTIMATOR,
        sensor_bound_m=SENSOR_BOUND_M,move_max_m=MOVE_MAX*LENGTH,force_max_N=FORCE_MAX,
        prior_intersection=True,linearization_error='analytic full Taylor callback',
        base_cost_weights=W_COST,constraint_names=CONSTRAINT_NAMES)
    (destination/'result.json').write_text(json.dumps(jsonable(RESULT),indent=2))
    (destination/'settings_and_sources.json').write_text(json.dumps(jsonable(dict(
        settings=settings,provenance=PROVENANCE,interface_audits=AUDIT_RESULTS)),indent=2))
    print('Saved:',destination)
if CLOSE_DRAKE_WORKER:
    BRIDGE.close()
    print('Drake worker closed. Rerun Cell 8 before another experiment.')


## What this notebook can establish after your validation

**Implemented connection:** the adaptive repository plans using a differentiable specialization
of Wei-Chen's smooth model and a justified smoothing-error envelope; the observation layer
actually calls his nonsmooth Drake system; the bounded-data estimator updates the parameter set.

**Additional integration work in this notebook:** coordinate conversion and process bridge;
prior-wide theorem envelope; checked numerical budget; nonlinear-parameter allowance;
analytic tube-remainder callback; required constraint ordering; estimator prior intersection;
strict solver/first-step/after-execution checks and error provenance.

**Reused, not reimplemented:** Drake's nonsmooth numerical solve and adaptive repository's
SQP/SLS/ADMM, observable-subspace factorization, generator update/reduction, and information reward.
`TrajOptProblem` and `IterativeTO` from Notebook 5 are not used here.

**Not yet established by construction alone:** successful convergence on your machine,
agreement with your installed Drake binary, improved learning/disturbance tradeoffs, or
real-world robustness. The numerical-error budget remains an assumption with per-transition
audits; the current information reward excludes horizon-coupled tube remainders. Conservative
envelopes can cause infeasibility or suppress learning. Record those outcomes rather than
turning off allowances to force a successful-looking run.

**Next tightening, if needed:** use the one-sided, state/input-dependent theorem generator
instead of its constant enclosing box, through a reviewed extension of the adaptive interface;
remove duplicate parameter-Taylor conservatism; then evaluate many masses/initial positions.
Frictional dynamics and noisy-state feedback require a separate model/estimator extension.
